# The hydrogen atom: orbitals and spectral lines

**Professor Nano Labs · Quantum foundations**

**Try it live in your browser:** [Hydrogen atom lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/hydrogen-atom/)
**Watch first:** [Schrodinger equation solutions to the hydrogen atom](https://www.youtube.com/watch?v=ag8tfER8-2g)
**Watch later:** [Spherical harmonics for Schrodinger equation](https://www.youtube.com/watch?v=yRJ2Xf2oH5w) · [Quantum orbital angular momentum](https://www.youtube.com/watch?v=9t0j4IQAJLA) · [How an electron binds two protons, the hydrogen molecule ion](https://www.youtube.com/watch?v=4i9YHKyNS9g)

An electron in the Coulomb potential $V(r)=-e^2/4\pi\varepsilon_0r$ of a proton held still. The wavefunctions separate into a radial part and a spherical harmonic,
$$\psi_{nlm}=R_{nl}(r)\,Y_l^m(\theta,\varphi),\qquad R_{nl}=\sqrt{\left(\frac{2}{na}\right)^3\frac{(n-l-1)!}{2n(n+l)!}}\,e^{-r/na}\left(\frac{2r}{na}\right)^lL_{n-l-1}^{2l+1}\!\left(\frac{2r}{na}\right),\qquad E_n=-\frac{13.6\text{ eV}}{n^2},$$
with $n=1,2,\dots$, $l=0,\dots,n-1$, $m=-l,\dots,l$ and the Bohr radius $a=0.529$ Å. We measure lengths in $a$ (so $a=1$ in the code).

The three sections match the lab's three tabs. It needs NumPy, SciPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import factorial
from scipy import special
from scipy.integrate import quad
from scipy.linalg import eigh_tridiagonal

RY = 13.605693122994        # eV
A0 = 0.529177210903         # Angstrom

def R(n, l, r):
    # the video's radial function, with r in units of a
    x = 2 * np.asarray(r, float) / n
    c = np.sqrt((2 / n) ** 3 * factorial(n - l - 1) / (2 * n * factorial(n + l)))
    return c * np.exp(-x / 2) * x ** l * special.eval_genlaguerre(n - l - 1, 2 * l + 1, x)

def Y(l, m, theta, phi):
    # complex spherical harmonic (Condon-Shortley phase); theta from +z, phi around z
    if hasattr(special, 'sph_harm_y'):
        return special.sph_harm_y(l, m, theta, phi)
    return special.sph_harm(m, l, phi, theta)

def Yreal(l, m, theta, phi):
    # the real orbitals, labelled as in the video's table: m > 0 -> cos(m phi) (p_x, d_xz, d_x2-y2), m < 0 -> sin (p_y, d_yz, d_xy)
    if m == 0:
        return Y(l, 0, theta, phi).real
    k = abs(m)
    y = Y(l, k, theta, phi)
    return np.sqrt(2) * (-1) ** k * (y.real if m > 0 else y.imag)

def psi(n, l, m, x, y, z, real=True):
    r = np.sqrt(x * x + y * y + z * z)
    theta = np.arccos(np.divide(z, r, out=np.ones_like(r), where=r > 0))
    phi = np.arctan2(y, x)
    return R(n, l, r) * (Yreal(l, m, theta, phi) if real else Y(l, m, theta, phi))

# the video's explicit radial functions (slide 10)
r = np.linspace(0, 30, 301)
slide10 = {(1, 0): 2 * np.exp(-r),
           (2, 0): (1 - r / 2) * np.exp(-r / 2) / np.sqrt(2),
           (2, 1): r * np.exp(-r / 2) / (2 * np.sqrt(6)),
           (3, 0): 2 / (3 * np.sqrt(3)) * (1 - 2 * r / 3 + 2 * r ** 2 / 27) * np.exp(-r / 3),
           (3, 1): 8 / (27 * np.sqrt(6)) * (1 - r / 6) * r * np.exp(-r / 3),
           (3, 2): 4 / (81 * np.sqrt(30)) * r ** 2 * np.exp(-r / 3)}
print('largest difference from the video\'s R_10 ... R_32:', max(np.max(np.abs(R(n, l, r) - f)) for (n, l), f in slide10.items()))
norm = max(abs(quad(lambda q: (q * R(n, l, q)) ** 2, 0, np.inf, limit=200)[0] - 1) for n in range(1, 6) for l in range(n))
print(f'int r^2 R_nl^2 dr = 1 for every n <= 5, l < n: worst error {norm:.1e}')
print(f'radial overlaps <2s|3s> = {quad(lambda q: q * q * R(2, 0, q) * R(3, 0, q), 0, np.inf)[0]:.1e}, <2p|4p> = {quad(lambda q: q * q * R(2, 1, q) * R(4, 1, q), 0, np.inf)[0]:.1e}')

## 1 · Orbitals

The probability density in the $xz$ plane, in the style of the video's pictures, for every orbital with $n\le3$. The real orbitals $\tilde\psi$ are the ones chemists draw; the complex ones with $m=\pm1$ and $\pm2$ have $|\psi|^2$ shaped like a doughnut around $z$. A brightness of $(|\psi|^2)^{0.4}$ brings out the faint parts.

In [ ]:
states = [(1, 0, 0, '1s'), (2, 0, 0, '2s'), (2, 1, 0, '2p$_z$'), (2, 1, 1, '2p$_x$'), (3, 0, 0, '3s'), (3, 1, 0, '3p$_z$'),
          (3, 1, 1, '3p$_x$'), (3, 2, 0, '3d$_{z^2}$'), (3, 2, 1, '3d$_{xz}$'), (3, 2, 2, '3d$_{x^2-y^2}$')]
fig, axs = plt.subplots(2, 5, figsize=(13, 6.4))
for ax, (n, l, m, lab) in zip(axs.flat, states):
    B = 4 * n * n + 2
    u = np.linspace(-B, B, 301)
    X, Z = np.meshgrid(u, u)
    d = np.abs(psi(n, l, m, X, 0 * X, Z)) ** 2
    ax.imshow((d / d.max()) ** 0.4, extent=[-B, B, -B, B], origin='lower', cmap='inferno')
    ax.set_title(f'{lab}  ({n},{l},{m})', fontsize=10); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('|psi|^2 on the xz plane (x across, z up)'); plt.tight_layout(); plt.show()

# nodes: n - l - 1 radial spheres at the zeros of L_{n-l-1}^{2l+1}(2r/n), l angular planes or cones
def radial_nodes(n, l):
    k = n - l - 1
    return n * special.roots_genlaguerre(k, 2 * l + 1)[0] / 2 if k else np.array([])
for n, l in [(2, 0), (3, 0), (3, 1), (4, 0)]:
    print(f'{n}{"spdf"[l]}: radial nodes at r = {np.round(radial_nodes(n, l), 4)} a')
print('3d_z2 angular nodes, 3cos^2(theta) = 1: theta =', np.round(np.degrees(np.arccos([1 / np.sqrt(3), -1 / np.sqrt(3)])), 2), 'degrees')

# complex m = +1 versus the real p_x
th, ph = np.meshgrid(np.linspace(0.1, 3, 7), np.linspace(0, 6, 9))
print('|Y_1^1|^2 depends on phi?', not np.allclose(np.abs(Y(1, 1, th, ph)) ** 2, np.abs(Y(1, 1, th, 0 * ph)) ** 2))
print('p_x = (Y_1^-1 - Y_1^1)/sqrt2:', np.allclose((Y(1, -1, th, ph) - Y(1, 1, th, ph)) / np.sqrt(2), Yreal(1, 1, th, ph)))

## 2 · The radial equation

With $u=rR$, the radial equation is a one-dimensional Schrödinger equation in the effective potential $V_\text{eff}=-2/r+l(l+1)/r^2$ (in Rydbergs, with $r$ in $a$). Solving it numerically on a grid recovers the Bohr energies without using the formula, for every $l$: the energy depends on $n$ alone. The plots repeat the video's slides 11 and 12.

In [ ]:
# finite differences: -u'' + V_eff u = (E / Ry) u on r = h, 2h, ..., u(0) = u(rmax) = 0
h, rmax = 0.01, 120.0
rg = np.arange(1, int(rmax / h)) * h
for l in range(3):
    diag = 2 / h ** 2 - 2 / rg + l * (l + 1) / rg ** 2
    E = eigh_tridiagonal(diag, -np.ones(len(rg) - 1) / h ** 2, select='i', select_range=(0, 3), eigvals_only=True) * RY
    exact = [-RY / (k + l + 1) ** 2 for k in range(4)]
    print(f'l = {l}: numerical {np.round(E, 4)}  Bohr {np.round(exact, 4)}  eV')

fig, axs = plt.subplots(1, 3, figsize=(13, 3.6))
r = np.linspace(0, 20, 600)
for n, l, c in [(1, 0, 'C3'), (2, 0, 'C0'), (2, 1, 'C0')]:
    axs[0].plot(r, R(n, l, r), color=c, ls='-' if l == 0 else '--', label=f'$R_{{{n}{l}}}$')
for l, ls in zip(range(3), ['-', '--', ':']):
    axs[1].plot(r, R(3, l, r), 'C2', ls=ls, label=f'$R_{{3{l}}}$')
for l, ls in zip(range(3), ['-', '--', ':']):
    rr = np.linspace(0, 30, 600)
    axs[2].plot(rr, (rr * R(3, l, rr)) ** 2, ls=ls, label=f'3{"spd"[l]}')
for ax in axs[:2]: ax.axhline(0, color='k', lw=0.6); ax.set_xlabel('r / a'); ax.legend()
axs[2].set(xlabel='r / a', ylabel='P(r) = r$^2$R$^2$'); axs[2].legend()
plt.tight_layout(); plt.show()

# the most likely and average distances, and the classically forbidden region
def peak(n, l):
    rr = np.linspace(1e-4, 4 * n * n + 10, 200001)
    return rr[np.argmax((rr * R(n, l, rr)) ** 2)]
def mean_r(n, l):
    return quad(lambda q: q ** 3 * R(n, l, q) ** 2, 0, np.inf, limit=200)[0]
def forbidden(n, l):
    s = np.sqrt(1 - l * (l + 1) / n ** 2); r1, r2 = n * n * (1 - s), n * n * (1 + s)
    P = lambda q: (q * R(n, l, q)) ** 2
    return quad(P, 0, r1)[0] + quad(P, r2, np.inf, limit=200)[0]
for n, l in [(1, 0), (2, 1), (3, 0), (3, 1), (3, 2)]:
    print(f'{n}{"spd"[l]}: most likely r = {peak(n, l):.3f} a, <r> = {mean_r(n, l):.3f} a (formula {(3 * n * n - l * (l + 1)) / 2}), outside the classical region {100 * forbidden(n, l):.1f}%')

## 3 · Spectral lines

A jump from $n_i$ to $n_f$ emits a photon with $hc/\lambda=E_{n_i}-E_{n_f}$, which is the Rydberg formula $1/\lambda=R(1/n_f^2-1/n_i^2)$. With the proton held still, $R=R_\infty=1.0974\times10^7$ m⁻¹. A real proton recoils a little: replacing $m_e$ by the reduced mass $m_em_p/(m_e+m_p)$ lengthens every wavelength by 0.054%, which brings H$\alpha$ to its measured 656.47 nm (vacuum).

In [ ]:
from scipy.constants import physical_constants as pc
Rinf = pc['Rydberg constant'][0] * 1e-9                       # 1/nm
mu = 1 / (1 + pc['electron-proton mass ratio'][0])            # reduced mass / m_e
lam = lambda ni, nf, Rc=Rinf: 1 / (Rc * (1 / nf ** 2 - 1 / ni ** 2))
names = {1: 'Lyman', 2: 'Balmer', 3: 'Paschen', 4: 'Brackett'}
for nf in range(1, 5):
    first = [lam(ni, nf) for ni in range(nf + 1, nf + 5)]
    print(f'{names[nf]:8s}: ' + ', '.join(f'{w:7.1f}' for w in first) + f' ...  limit {lam(np.inf, nf):7.1f} nm')
print(f'H-alpha: {lam(3, 2):.2f} nm with the proton held still, {lam(3, 2, Rinf * mu):.2f} nm with the reduced mass (measured: 656.47 nm in vacuum)')

fig, ax = plt.subplots(figsize=(11, 2.4))
for nf, c in zip(range(1, 5), ['C1', 'C2', 'C0', 'C3']):
    E = [RY * (1 / nf ** 2 - 1 / ni ** 2) for ni in range(nf + 1, nf + 31)]
    ax.vlines(E, 0, 1, color=c, lw=1, label=names[nf])
ax.axvspan(1240 / 750, 1240 / 380, color='gold', alpha=0.25, label='visible')
ax.set(xscale='log', xlabel='photon energy (eV)', yticks=[], ylim=(0, 1.55)); ax.legend(ncol=5, loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

## Check the lab's answers

These asserts confirm the numbers in the lab's "Predict first" answers.

In [ ]:
# Q1: the 2s node is a sphere at r = 2a, positive inside and negative outside
assert np.allclose(radial_nodes(2, 0), [2.0]) and R(2, 0, 1) > 0 > R(2, 0, 3)
# Q2: 3d_z2 has no radial node and two cones at 54.7 and 125.3 degrees
assert len(radial_nodes(3, 2)) == 0 and np.allclose(np.degrees(np.arccos([1 / np.sqrt(3), -1 / np.sqrt(3)])), [54.7356, 125.2644], atol=1e-3)
# Q3: |psi_{2,1,1}|^2 does not depend on phi, and p_x = (psi_{2,1,-1} - psi_{2,1,1}) / sqrt2
x, y, z = np.random.default_rng(1).normal(size=(3, 50)) * 4
assert np.allclose(np.abs(psi(2, 1, 1, x, y, z, False)) ** 2, np.abs(psi(2, 1, 1, np.hypot(x, y), 0 * y, z, False)) ** 2)
assert np.allclose((psi(2, 1, -1, x, y, z, False) - psi(2, 1, 1, x, y, z, False)) / np.sqrt(2), psi(2, 1, 1, x, y, z, True))
# Q4: for 1s the density peaks at the nucleus, but the most likely distance is a, and <r> = 1.5a
assert abs(peak(1, 0) - 1) < 1e-3 and abs(mean_r(1, 0) - 1.5) < 1e-9
# Q5: 13 e^-4 = 23.8% of the 1s probability lies beyond r = 2a
assert abs(forbidden(1, 0) - 13 * np.exp(-4)) < 1e-9 and round(100 * forbidden(1, 0), 1) == 23.8
# Q6: <r> = 13.5, 12.5, 10.5 a for 3s, 3p, 3d; 3d peaks at 9a; only s is nonzero at the nucleus
assert np.allclose([mean_r(3, l) for l in range(3)], [13.5, 12.5, 10.5]) and abs(peak(3, 2) - 9) < 1e-3
assert R(3, 0, 0) > 0 and R(3, 1, 0) == 0 and R(3, 2, 0) == 0
# Q7: H-alpha is 3 -> 2 at 656.1 nm; the reduced mass moves it to 656.47 nm (0.054%)
assert round(lam(3, 2), 1) == 656.1 and abs(lam(3, 2, Rinf * mu) - 656.47) < 0.01
# Q8: the Lyman limit is 1/R = 91.1 nm, a 13.6 eV photon
assert round(lam(np.inf, 1), 1) == 91.1 and round(RY, 1) == 13.6
print('all the lab\'s answers check out')

## Things to try

1. **Hybrid orbitals.** Add $2s$ and $2p_z$ with equal weight. Where is the density now, and why does it point one way? This $sp$ hybrid is how carbon forms directional bonds.
2. **The Zeeman effect.** In a magnetic field $B$ along $z$, the energy shifts by $\mu_BBm$. Which orbitals split, and why are the complex orbitals the natural ones here?
3. **Other one-electron atoms.** For He⁺ or Li²⁺, replace $e^2$ by $Ze^2$. How do the size, the energies and the wavelengths scale with $Z$?
4. **Transition strengths.** Compute $\langle n'l'm'|z|nlm\rangle$ for a few pairs. Which are zero? You will find the selection rule $\Delta l=\pm1$.